In [1]:
#working on the train_set
#importing the libraries
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
print("Libraries imported successfully")


Libraries imported successfully


In [2]:

#now we are converting our images to csv format that are already in jpg
import os
from PIL import Image

# Path to your "test_set" folder
image_dir = 'test_set'
data = []


label_map = {'cats': 0, 'dogs': 1}  # Map folder names to labels

for label_name in os.listdir(image_dir):
    label_path = os.path.join(image_dir, label_name)
    
    if os.path.isdir(label_path) and label_name in label_map:
        label = label_map[label_name]
        
        for file in os.listdir(label_path):
            if file.endswith('.jpg'):
                img_path = os.path.join(label_path, file)
                
                try:
                    image = Image.open(img_path).convert('L')  # grayscale
                    image = image.resize((28, 28))             # resize to 28x28
                    pixels = np.array(image).flatten()         # flatten to 784
                    row = [label] + pixels.tolist()
                    data.append(row)
                except:
                    print(f"Error reading {img_path}")

# Create column names: label, pixel0, pixel1, ..., pixel783
columns = ['label'] + [f'pixel{i}' for i in range(784)]

# Convert to DataFrame and save
df = pd.DataFrame(data, columns= columns )
df.to_csv('cats_vs_dogs(test).csv', index=False)



In [3]:
raw_df = pd.read_csv('cats_vs_dogs(test).csv')
X_test= raw_df.drop('label', axis=1).values
y_test = raw_df['label'].values




In [4]:




#now we will create a custom dataset class
class CatsVsDogsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype =torch.float32)
        self.y= torch.tensor(y,dtype=torch.long)

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, index):

        return self.X[index], self.y[index]
    
# Create the dataset
test_dataset = CatsVsDogsDataset(X_test, y_test, )
# Create the DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)




In [5]:
import torch
import torch.nn as nn

# Define the same model structure
class SimpleNN(nn.Module):
    def __init__(self, n_features=784, n_classes=2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.model(x)

# Create model instance
model = SimpleNN(n_features=X_test.shape[1])  # match same input size
# Load the trained model state
model.load_state_dict(torch.load('cats_vs_dogs_model.pth'))  # Load the saved model weights
model.eval()  # Set the model to evaluation mode

SimpleNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [6]:
total = 0
correct = 0
# Iterate over the test dataset
with torch.no_grad():
    for inputs,labels in test_loader:
        outputs = model(inputs)
        _,predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')    



Test Accuracy: 59.00%
